In [1]:
from pdfminer.high_level import extract_text
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup
import re
import spacy, json

#### PDF OCR

In [2]:
text = extract_text("data/Draft 12_15_22 - Project Hail Mary Screenplay.pdf")
with open("phm_script_raw_draft.txt", "w", encoding="utf-8") as f:
    f.write(text)

#### EPUB to txt

In [12]:
BLOCK_TAGS = {"p", "div", "br", "h1", "h2", "h3", "h4", "h5", "h6", "li", "tr"}

def epub_to_text(epub_path):
    book = epub.read_epub(epub_path)
    chapters = []

    for item in book.get_items():
        if item.get_type() == ebooklib.ITEM_DOCUMENT:
            soup = BeautifulSoup(item.get_content(), "html.parser")

            # Insert newline markers at block boundaries only
            for tag in soup.find_all(BLOCK_TAGS):
                tag.insert_before("\n")
                tag.insert_after("\n")

            # Now get_text with no separator — inline tags just yield their text naturally
            text = soup.get_text(separator="")

            # Clean up
            text = re.sub(r" {2,}", " ", text)        # collapse extra spaces
            text = re.sub(r"\n{3,}", "\n\n", text)    # collapse extra blank lines
            text = re.sub(r" \n", "\n", text)         # trailing spaces before newlines
            text = re.sub(r"\n ", "\n", text)         # leading spaces after newlines

            chapters.append(text.strip())

    return "\n\n".join(chapters)

full_text = epub_to_text("data/Project Hail Mary.epub")
with open("book_raw.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

In [3]:
import re
import json

OPEN_QUOTE  = "\u201c"
CLOSE_QUOTE = "\u201d"
ATTRIBUTION = re.compile(
    r',?\s*(he says|Rocky says|he asks|Rocky asks|Rocky calls|he adds'
    r'|he says again|Rocky says out of nowhere|he asks again)[^"]*',
    re.IGNORECASE,
)

def extract_speech(line: str) -> str:
    fragments = re.findall(
        rf'{OPEN_QUOTE}([^{CLOSE_QUOTE}]+){CLOSE_QUOTE}|"([^"]+)"',
        line
    )
    if fragments:
        parts = [a or b for a, b in fragments]
        parts = [p.rstrip(", ") for p in parts]
        return " ".join(parts).strip()
    cleaned = ATTRIBUTION.sub(" ", line)
    return re.sub(r"\s+", " ", cleaned).strip()

with open("rocky_book_lines.txt", encoding="utf-8") as f:
    raw_lines = [l.rstrip("\n") for l in f if l.strip()]

# Clean each line first
cleaned_lines = [extract_speech(l) for l in raw_lines]

# Build exact vocab — just tokenize on whitespace and punctuation
vocab = set()
for line in cleaned_lines:
    tokens = re.findall(r"[^\s.,!?;:\"\'""''…—\-]+", line)
    for t in tokens:
        vocab.add(t)  # exact form, no lowercasing, no lemmatization

# Save vocab
with open("rocky_vocab.json", "w", encoding="utf-8") as f:
    json.dump(sorted(vocab), f, indent=2, ensure_ascii=False)

# Save cleaned lines for inspection
with open("rocky_lines_clean.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(cleaned_lines))

print(f"Total unique tokens: {len(vocab)}")
print("\nSample vocab:")
print(sorted(vocab)[:50])

Total unique tokens: 1332

Sample vocab:
['0', '1', '100', '12', '15', '198', '20', '265', '35', '4', '42', '48', '5', '6', '61', '64', '8', '80', '82', '86', '928', 'A', 'About', 'Accelerate', 'Adjust', 'Adrian', 'After', 'Again', 'Agree', 'Ah', 'Aim', 'Air', 'All', 'Also', 'Alternate', 'Always', 'Am', 'Amaze', 'And', 'Anger', 'Angry', 'Another', 'Any', 'Apologies', 'Apology', 'Approximately', 'Ask', 'Astronomy', 'Astrophage', 'Attempted']


<>:33: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
<>:33: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
/var/folders/ln/ccc333gd343_2blclyk9w8wc0000gn/T/ipykernel_52232/4191262702.py:33: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
  tokens = re.findall(r"[^\s.,!?;:\"\'""''…—\-]+", line)
